# Dynamic Programming

Dynamic programming (DP) is a family of algorithms for solving a finite Markov decision process when its complete model $p(s',r\mid s,a)$ is known. Unlike model-free reinforcement learning, DP can compute expected updates directly from every possible transition and reward.

We assume finite state and action spaces, the Markov property, and either $\gamma<1$ or guaranteed episode termination. For episodic tasks, an absorbing terminal state is included with value $V(\text{terminal})=0$.

DP repeatedly applies Bellman backups to propagate value information through the state space. Its two central problems are:

- **Prediction:** compute $v_\pi$ for a fixed policy $\pi$.
- **Control:** find an optimal value function and an optimal policy.

## Policy Evaluation (Prediction)

Given a fixed policy $\pi$, policy evaluation computes its state-value function $v_\pi$. It is a **prediction** problem: the policy is evaluated, not changed.

$$v_\pi(s)=\sum_a\pi(a\mid s)\sum_{s',r}p(s',r\mid s,a)\left[r+\gamma v_\pi(s')\right].$$

Because $v_\pi$ is initially unknown, start from an arbitrary estimate $V_0$ and repeatedly apply the Bellman expectation backup:

$$V_{k+1}(s)\leftarrow\sum_a\pi(a\mid s)\sum_{s',r}p(s',r\mid s,a)\left[r+\gamma V_k(s')\right].$$

Each update averages over every action selected by $\pi$ and every possible model outcome. Repeated sweeps propagate successor values backward until $V$ approaches the fixed point $v_\pi$. Convergence holds for discounted tasks with $\gamma<1$ and for terminating episodic tasks under standard conditions.

### Iterative Policy Evaluation

```text
Input: policy pi, model p, discount gamma, tolerance theta
Initialize V(s) arbitrarily; set V(terminal) = 0

repeat
    delta = 0
    for each nonterminal state s:
        old_value = V(s)
        V(s) = sum_a pi(a|s) * sum_{s',r} p(s',r|s,a)
               * [r + gamma * V(s')]
        delta = max(delta, |old_value - V(s)|)
until delta < theta

return V
```

This is an **in-place** implementation: values updated earlier in a sweep may be used immediately by later updates. $\Delta$ measures the largest state-value change and provides the stopping criterion.

## Policy Improvement

After evaluating $\pi$, we use $v_\pi$ to determine whether another action is better. For each state-action pair, compute the value of taking $a$ once and then following the current policy:

$$q_\pi(s,a)=\sum_{s',r}p(s',r\mid s,a)\left[r+\gamma v_\pi(s')\right].$$

For a deterministic policy, continuing with its current action has value

$$v_\pi(s)=q_\pi(s,\pi(s)).$$

Therefore, if an alternative satisfies

$$q_\pi(s,a)>v_\pi(s),$$

taking $a$ in state $s$ and then following $\pi$ is better than following $\pi$ immediately. Replacing the current action by the best one gives the greedy policy

$$\begin{aligned}
\pi'(s)
&=\arg\max_a q_\pi(s,a)\\
&=\arg\max_a\sum_{s',r}p(s',r\mid s,a)
\left[r+\gamma v_\pi(s')\right].
\end{aligned}$$

The **policy improvement theorem** states that if

$$q_\pi(s,\pi'(s))\geq v_\pi(s)\qquad\forall s,$$

then

$$v_{\pi'}(s)\geq v_\pi(s)\qquad\forall s.$$

Why this is global rather than only one-step improvement: after choosing the improved action, the same inequality can be applied again at the successor state. Repeated substitution yields the complete return generated by $\pi'$, proving that $\pi'$ cannot be worse than $\pi$.

### Greedy Policy Improvement

```text
Input: evaluated values V approximately equal to v_pi, model p

policy_stable = true
for each nonterminal state s:
    old_action = pi(s)
    pi(s) = argmax_a sum_{s',r} p(s',r|s,a)
            * [r + gamma * V(s')]

    if pi(s) != old_action:
        policy_stable = false

return pi, policy_stable
```

Policy evaluation asks, **how good is $\pi$?** Policy improvement asks, **which action is greedy with respect to $v_\pi$?** If the greedy policy is unchanged, $\pi$ satisfies the Bellman optimality equation and is optimal.

## Policy Iteration

Policy iteration alternates the two previous procedures:

$$\pi_0\xrightarrow{\text{evaluate}}v_{\pi_0}
\xrightarrow{\text{improve}}\pi_1
\xrightarrow{\text{evaluate}}v_{\pi_1}
\xrightarrow{\text{improve}}\cdots$$

One **outer iteration** consists of:

1. **Policy evaluation:** hold $\pi_k$ fixed and iteratively compute $V\approx v_{\pi_k}$.
2. **Policy improvement:** hold $V$ fixed and greedily update the action at every state:

$$\pi_{k+1}(s)=\arg\max_a\sum_{s',r}p(s',r\mid s,a)
\left[r+\gamma V(s')\right].$$

These phases are not combined inside each individual state-value update. Evaluation first makes $V$ consistent with the current policy; improvement then constructs a new policy from those values.

### Algorithm

```text
Initialize V(s) arbitrarily and pi(s) arbitrarily
Set V(terminal) = 0

repeat
    # 1. Policy evaluation
    repeat
        delta = 0
        for each nonterminal state s:
            old_value = V(s)
            V(s) = sum_{s',r} p(s',r|s,pi(s))
                   * [r + gamma * V(s')]
            delta = max(delta, |old_value - V(s)|)
    until delta < theta

    # 2. Policy improvement
    policy_stable = true
    for each nonterminal state s:
        old_action = pi(s)
        pi(s) = argmax_a sum_{s',r} p(s',r|s,a)
                * [r + gamma * V(s')]
        if pi(s) != old_action:
            policy_stable = false

until policy_stable

return V, pi
```

Each improvement produces a policy at least as good as the previous one. In a finite MDP, the process terminates after finitely many policy changes. A stable policy is optimal.

## Value Iteration

Policy iteration evaluates the current policy through many sweeps before improving it. Value iteration truncates policy evaluation to **one sweep** and performs improvement inside every state update.

Instead of backing up the action selected by a fixed policy,

$$V_{k+1}(s)\leftarrow
\sum_{s',r}p(s',r\mid s,\pi(s))
\left[r+\gamma V_k(s')\right],$$

value iteration immediately selects the action with the largest backup:

$$V_{k+1}(s)\leftarrow
\max_a\sum_{s',r}p(s',r\mid s,a)
\left[r+\gamma V_k(s')\right].$$

This is the Bellman optimality backup. The estimate $V_k$ need not equal the value of any single fixed policy: each update acts greedily with respect to the current successor estimates. Repeated backups propagate rewards backward while continually revising which action appears best. Under the standard finite-MDP conditions, $V_k\to v_*$. 

The conceptual difference is therefore:

- **Policy iteration:** evaluate one policy accurately, then improve it.
- **Value iteration:** partially evaluate and immediately improve at every sweep.

### Algorithm

```text
Input: model p, discount gamma, tolerance theta
Initialize V(s) arbitrarily; set V(terminal) = 0

repeat
    delta = 0
    for each nonterminal state s:
        old_value = V(s)
        V(s) = max_a sum_{s',r} p(s',r|s,a)
               * [r + gamma * V(s')]
        delta = max(delta, |old_value - V(s)|)
until delta < theta

for each nonterminal state s:
    pi(s) = argmax_a sum_{s',r} p(s',r|s,a)
            * [r + gamma * V(s')]

return V, pi
```

The $\max$ updates the value estimate; the final $\arg\max$ extracts the action that produced that value. Thus, value iteration can compute $v_*$ without explicitly storing a policy during convergence.

## Asynchronous Dynamic Programming

Standard DP uses **sweeps**: every state is updated once before the next sweep starts. This is expensive when $|\mathcal S|$ is large and may waste computation on states that are currently irrelevant.

Asynchronous DP removes the sweep requirement. At each computation step, it selects any state $s_k$ and applies an in-place Bellman backup only to that state:

$$V(s_k)\leftarrow
\max_a\sum_{s',r}p(s',r\mid s_k,a)
\left[r+\gamma V(s')\right].$$

The backup equation is identical to value iteration. Only the **update schedule** changes:

- states may be updated in any order;
- some states may be updated more frequently than others;
- updated values are immediately available to later backups;
- a complete sweep is not required before useful progress occurs.

For convergence, every state whose value matters must continue to be selected; no relevant state can be ignored forever. Prioritizing states with large value changes or states currently visited by the agent can propagate useful information faster.

### Asynchronous Value Iteration

```text
Initialize V(s) arbitrarily; set V(terminal) = 0

while computation continues:
    select a nonterminal state s
    V(s) = max_a sum_{s',r} p(s',r|s,a)
           * [r + gamma * V(s')]

for each nonterminal state s:
    pi(s) = argmax_a sum_{s',r} p(s',r|s,a)
            * [r + gamma * V(s')]
```

Asynchronous DP is useful for real-time planning: computation can focus on states relevant to the agent's current trajectory while the latest value estimates guide behavior.

## Generalized Policy Iteration

Generalized policy iteration (GPI) is the interaction of two processes:

1. **Evaluation** moves the value estimate toward consistency with the current policy:

$$V\longrightarrow v_\pi.$$

2. **Improvement** moves the policy toward greediness with respect to the current value estimate:

$$\pi\longrightarrow\operatorname{Greedy}(V).$$

The processes do not need to finish or alternate in rigid phases. They may use complete sweeps, partial sweeps, or individual asynchronous updates. Policy iteration performs near-complete evaluation before improvement; value iteration interleaves them at every backup.

Evaluation and improvement temporarily oppose each other. Changing $\pi$ makes $V$ inconsistent with the new policy; changing $V$ may change which policy is greedy. Nevertheless, both processes move toward a joint fixed point.

They stabilize simultaneously when

$$V=v_\pi$$

and

$$\pi=\operatorname{Greedy}(V).$$

Combining these conditions gives

$$V(s)=\max_a\sum_{s',r}p(s',r\mid s,a)
\left[r+\gamma V(s')\right],$$

which is the Bellman optimality equation. Therefore, the joint fixed point is $V=v_*$ with an optimal policy $\pi_*$. Most reinforcement-learning control algorithms are instances of GPI with approximate evaluation and improvement.

## Efficiency of Dynamic Programming

Exact DP requires a known model and storage proportional to the state space. A full sweep updates every state, so large $|\mathcal S|$ can make synchronous methods impractical. However, DP is substantially more efficient than enumerating all deterministic policies, whose number grows exponentially with the number of states.

In practice, good initialization, in-place updates, prioritized state selection, and asynchronous backups can reduce computation. For large problems, updates can focus on states reachable along relevant or optimal trajectories rather than treating every state equally.